In [0]:
# ============================================================
# MODULE 2 - SCHEMA ENFORCEMENT
# ============================================================

table_name = "workspace.my_databricks_demo.transactions"

spark.table(table_name).printSchema()

In [0]:
# ============================================================
# CREATE DATAFRAME WITH NEW COLUMN
# ============================================================

from pyspark.sql.functions import lit
from pyspark.sql.types import StringType

df_base = spark.table(table_name)

df_with_promo = df_base.withColumn("promo_code", lit("PROMO10").cast(StringType()))

df_with_promo.printSchema()

display(df_with_promo)

In [0]:
# ============================================================
# SCHEMA EVOLUTION
# ============================================================

df_with_promo.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(table_name)

In [0]:
# ============================================================
# VERIFY NEW COLUMN
# ============================================================

spark.table(table_name).printSchema()

display(spark.table(table_name))

In [0]:
# ============================================================
# INCOMPATIBLE DATA TYPE
# ============================================================

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

bad_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("store_id", StringType(), True),
    StructField("transaction_date", DateType(), True),
    StructField("quantity", StringType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("promo_code", StringType(), True)
])

bad_data = [
    ("T900", "C900", "P900", "S900", None, "abc", 100.0, 500.0, "PROMO20")
]

bad_df = spark.createDataFrame(bad_data, schema=bad_schema)

bad_df.printSchema()

In [0]:
# ============================================================
# TEST INCOMPATIBLE SCHEMA
# ============================================================

bad_df.write.format("delta").mode("append").saveAsTable(table_name)

In [0]:
# ============================================================
# TABLE HISTORY
# ============================================================

spark.sql(f"DESCRIBE HISTORY {table_name}").show(truncate=False)

In [0]:
# ============================================================
# FINAL TABLE
# ============================================================

display(spark.table(table_name))

testing with another int value with string datatype

In [0]:
# ============================================================
# INCOMPATIBLE DATA TYPE
# ============================================================

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

bad_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("store_id", StringType(), True),
    StructField("transaction_date", DateType(), True),
    StructField("quantity", StringType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("promo_code", StringType(), True)
])

bad_data = [
    ("T1000", "C1000", "P1000", "S1000", None, "6", 150.0, 300.0, "PROMO20")
]

bad_df = spark.createDataFrame(bad_data, schema=bad_schema)

bad_df.printSchema()

In [0]:
# ============================================================
# TEST INCOMPATIBLE SCHEMA
# ============================================================

bad_df.write.format("delta").mode("append").saveAsTable(table_name)

its saved

In [0]:


# ============================================================
# CHECK TEST ROW
# ============================================================

display(spark.table(table_name).filter("transaction_id = 'T1000'"))

In [0]:
# ============================================================
# REMOVE TEST ROW
# ============================================================

spark.sql(f"DELETE FROM {table_name} WHERE transaction_id = 'T900'")

In [0]:
%sql

select * from workspace.my_databricks_demo.transactions;

In [0]:
# ============================================================
# TEST 2 - MISSING REQUIRED COLUMN--
# ============================================================

from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType, IntegerType

missing_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("store_id", StringType(), True),
    StructField("transaction_date", DateType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True)
])

missing_data = [
    ("T901", "C901", "P901", "S901", None, 2, 50.0)
]

missing_df = spark.createDataFrame(missing_data, schema=missing_schema)

missing_df.printSchema()

missing_df.write.format("delta").mode("append").saveAsTable(table_name)

In [0]:
%sql

select * from workspace.my_databricks_demo.transactions;

===================
 TEST 2 - A missing column in the incoming DataFrame does not necessarily cause a schema error; the existing table column can receive NULL.
 =======================

In [0]:
# ============================================================
# TEST 3 - MISSING REQUIRED COLUMN without schema defintion--
# ============================================================

missing_data = [ ("T1101", "C1101", "P1101", "S1101",None, 2, 50.0)]

missing_df = spark.createDataFrame(missing_data)

missing_df.printSchema()

missing_df.write.format("delta").mode("append").saveAsTable(table_name)

In [0]:
# ============================================================
# TEST 4 - 1  COLUMN without schema defintion--
# ============================================================

from datetime import date

one_data = [ ("T1101", "C1101", "P1101", "S1101", date(2026, 9, 13), 2, 50.0,100)]

one_data = spark.createDataFrame(one_data)

one_data.printSchema()

one_data.write.format("delta").mode("append").saveAsTable(table_name)

In [0]:
# ============================================================
# TEST 5- all COLUMN without schema defintion-- schema inference
# ============================================================

from datetime import date

all_data = [ ("T1101", "C1101", "P1101", "S1101", date(2026, 9, 13), 2, 50.0,100,None)]

all_data = spark.createDataFrame(all_data)

all_data.printSchema()

all_data.write.format("delta").mode("append").saveAsTable(table_name)

In [0]:
# ============================================================
# TEST 5.1- all COLUMN without schema defintion-- schema inference
# ============================================================

from datetime import date

all_data = [ ("T1101", "C1101", "P1101", "S1101", date(2026, 9, 13), 2, 50.0,100.0,None)]
columns = [
    "transaction_id",
    "customer_id",
    "product_id",
    "store_id",
    "transaction_date",
    "quantity",
    "unit_price",
    "total_amount",
    "promo_code"
]
all_data = spark.createDataFrame(all_data,columns)

all_data.printSchema()

all_data.write.format("delta").mode("append").saveAsTable(table_name)

In [0]:


# ============================================================
# CHECK TEST ROW
# ============================================================

display(spark.table(table_name).filter("transaction_id = 'T900'"))

In [0]:
# ============================================================
# TEST 6 - all COLUMN without schema defintion--
# ============================================================

from datetime import date
columns = [
    "transaction_id",
    "customer_id",
    "product_id",
    "store_id",
    "transaction_date",
    "quantity",
    "unit_price",
    "total_amount",
    "promo_code"
]
all_data = [ ("T1101", "C1101", "P1101", "S1101", date(2026, 9, 13), 2, 50.0,100.0,"promo")]

all_data = spark.createDataFrame(all_data,columns)

all_data.printSchema()

all_data.write.format("delta").mode("append").saveAsTable(table_name)

In [0]:
%sql

select * from workspace.my_databricks_demo.transactions;

In [0]:
from pyspark.sql.functions import  lit

extra_df = spark.table(table_name).withColumn("source_system", lit("WEB"))

extra_df.printSchema()

extra_df.write.format("delta").mode("append").saveAsTable(table_name)

In [0]:
from pyspark.sql.functions import  lit

data = [
    ("T1201", "C1201", "P1201", "S1201", date(2026, 9, 14), 3, 20.0, 60.0, "promo", "WEB")
]

extra_df = spark.createDataFrame(data, columns)

extra_df.printSchema()

extra_df.write.format("delta").mode("append").saveAsTable(table_name)

In [0]:
# FINAL SCHEMA CHECK

spark.table(table_name).printSchema()

display(spark.table(table_name))

In [0]:


# TABLE HISTORY

spark.sql(f"DESCRIBE HISTORY {table_name}").show(truncate=False)